In [ ]:
import kagglehub
import os
from google.colab import files
import json
import random
import shutil
import matplotlib.pyplot as plt
import torch
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.models import mobilenet_v2
from torchvision.models import MobileNet_V2_Weights
import torch.nn as nn

In [ ]:
path = kagglehub.dataset_download("kedarsai/bird-species-classification-220-categories")


train_path = f"{path}/Train"
test_path = f"{path}/Test"

random.seed(42)
birds50_train = "/content/Birds50/Train"
birds50_test = "/content/Birds50/Test"

os.makedirs(birds50_train, exist_ok=True)
os.makedirs(birds50_test, exist_ok=True)

all_classes = sorted(os.listdir(train_path))

selected_classes = random.sample(all_classes,50)

print("Selected Classes:", len(selected_classes))

for cls in selected_classes:
    shutil.copytree(
        os.path.join(train_path, cls),
        os.path.join(birds50_train, cls),
        dirs_exist_ok=True
    )
    shutil.copytree(
        os.path.join(test_path, cls),
        os.path.join(birds50_test, cls),
        dirs_exist_ok=True
    )
    
train_path = "/content/Birds50/Train"
test_path = "/content/Birds50/Test"
classes = sorted(os.listdir(train_path))

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET), 
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    root=train_path,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    root=test_path,
    transform=transform
)

print("Train Images:", len(train_dataset))
print("Test Images:", len(test_dataset))
print("Classes:", len(train_dataset.classes))

image, label = train_dataset[0]

train_path = "/content/Birds50/Train"
test_path = "/content/Birds50/Test"

print(len(train_dataset.classes))

In [ ]:
plt.imshow(
    image.permute(1, 2, 0)
)
plt.show()

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

In [ ]:
weights = MobileNet_V2_Weights.DEFAULT

model = mobilenet_v2(
    weights=weights
)

num_classes = len(
    train_dataset.classes
)

model.classifier[1] = nn.Linear(
    in_features=1280,
    out_features=num_classes
)
print(model.classifier)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print(device)
model = model.to(device)

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,          # Slightly lower baseline LR prevents destroying pre-trained weights
    weight_decay=1e-2 # Regularization prevents overfitting
)
trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Trainable: {trainable:,}")
print(f"Total: {total:,}")

In [ ]:
def evaluate(model, dataloader, device):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in dataloader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = (100 * correct / total)
    return accuracy

 

In [ ]:
EPOCHS=10
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)
train_losses=[]
best_acc = 0
for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    epoch_loss = (running_loss/len(train_loader))
    train_losses.append(epoch_loss)
    val_acc = evaluate(model,test_loader,device)

    if val_acc > best_acc:
      best_acc = val_acc

      torch.save(model.state_dict(),"birds50_best.pth")
    print(
    f"Epoch {epoch+1}/{EPOCHS}"
    f" | LR: {optimizer.param_groups[0]['lr']:.7f}"
    f" | Loss: {epoch_loss:.4f}"
    f" | Val Acc: {val_acc:.2f}%"
    )
    scheduler.step()
    

In [ ]:
with open(
    "class_names.json",
    "w"
) as f:
    json.dump(
        train_dataset.classes,
        f
    )
with open(
    "selected_classes.json",
    "w"
) as f:
    json.dump(
        selected_classes,
        f
    )
files.download("class_names.json")
files.download("selected_classes.json")
files.download("birds50_best.pth")